In [1]:
"""
Lecture 2.3 Supplemental Code

Jackson School of Global Affairs                 
                                                 
 Created by Ardina Hasanbasri for GLBL 5021      
                                                 
Additional reference code and data used:        
Cunningham (2020)      
https://mixtape.scunning.com/07-instrumental_variables#college-in-the-county              
                                            
"""

'\nLecture 2.3 Supplemental Code\n\nJackson School of Global Affairs                 \n\n Created by Ardina Hasanbasri for GLBL 5021      \n\nAdditional reference code and data used:        \nCunningham (2020)      \nhttps://mixtape.scunning.com/07-instrumental_variables#college-in-the-county              \n\n'

In [3]:
#------------------------------------------------#
# Setting Up and Load Data                       #
# -----------------------------------------------#

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
import pyreadstat

In [4]:
#---------------------------------------------------------
# Example 1: How do prices affect cigarette consumption? 
#---------------------------------------------------------

# Let's load the data!

url = "https://vincentarelbundock.github.io/Rdatasets/csv/AER/CigarettesSW.csv"
cigs = pd.read_csv(url)

# Keep 1995 only (like your R subset)
cigs = cigs[cigs["year"] == 1995].copy()


In [6]:
#------------------------------------------------#
# Some Data Cleaning                           #
# -----------------------------------------------#

# Real price
cigs["rprice"] = cigs["price"] / cigs["cpi"]

# Real income per capita
cigs["rincome"] = cigs["income"] / cigs["population"] / cigs["cpi"]

# Tax difference instrument
cigs["tdiff"] = (cigs["taxs"] - cigs["tax"]) / cigs["cpi"]

# Log variables
cigs["log_packs"] = np.log(cigs["packs"])
cigs["log_rprice"] = np.log(cigs["rprice"])
cigs["log_rincome"] = np.log(cigs["rincome"])
cigs["tax_cpi"] = cigs["tax"] / cigs["cpi"]


In [8]:
#-------------------------------------
# Q1) What would OLS result look like? 
#-------------------------------------

ols = smf.ols("log_packs ~ log_rprice + log_rincome", data=cigs).fit(cov_type="HC1")
print(ols.summary())


                            OLS Regression Results                            
Dep. Variable:              log_packs   R-squared:                       0.433
Model:                            OLS   Adj. R-squared:                  0.408
Method:                 Least Squares   F-statistic:                     18.37
Date:                Thu, 19 Feb 2026   Prob (F-statistic):           1.47e-06
Time:                        12:37:29   Log-Likelihood:                 13.840
No. Observations:                  48   AIC:                            -21.68
Df Residuals:                      45   BIC:                            -16.07
Df Model:                           2                                         
Covariance Type:                  HC1                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      10.3420      0.966     10.701      

In [9]:
#-------------------------------------
# Q2) What would IV reg look like?  
#-------------------------------------

iv_model = IV2SLS.from_formula("log_packs ~ 1 + log_rincome + [log_rprice ~ tdiff + tax_cpi]", data=cigs).fit(cov_type="robust")
print(iv_model.summary)


                          IV-2SLS Estimation Summary                          
Dep. Variable:              log_packs   R-squared:                      0.4294
Estimator:                    IV-2SLS   Adj. R-squared:                 0.4041
No. Observations:                  48   F-statistic:                    34.506
Date:                Thu, Feb 19 2026   P-value (F-stat)                0.0000
Time:                        12:40:30   Distribution:                  chi2(2)
Cov. Estimator:                robust                                         
                                                                              
                              Parameter Estimates                              
             Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------
Intercept       9.8950     0.9288     10.654     0.0000      8.0746      11.715
log_rincome     0.2804     0.2458     1.1407    

In [ ]:
#-------------------------------------
# Q3) Manually do Stage 1 and Stage 2  
#-------------------------------------

# We can get similar results (except for standard error):

stage1 = smf.ols("log_rprice ~ log_rincome + tdiff + tax_cpi", data=cigs).fit(cov_type="HC1")
print(stage1.summary())


                            OLS Regression Results                            
Dep. Variable:             log_rprice   R-squared:                       0.940
Model:                            OLS   Adj. R-squared:                  0.936
Method:                 Least Squares   F-statistic:                     263.1
Date:                Wed, 18 Feb 2026   Prob (F-statistic):           4.14e-28
Time:                        22:51:56   Log-Likelihood:                 98.806
No. Observations:                  48   AIC:                            -189.6
Df Residuals:                      44   BIC:                            -182.1
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       4.1030      0.088     46.425      

In [ ]:
# Predict after the first stage. 

cigs["predicted_log_rprice"] = stage1.fittedvalues


In [9]:
stage2 = smf.ols("log_packs ~ predicted_log_rprice + log_rincome", data=cigs).fit(cov_type="HC1")
print(stage2.summary())


                            OLS Regression Results                            
Dep. Variable:              log_packs   R-squared:                       0.337
Model:                            OLS   Adj. R-squared:                  0.307
Method:                 Least Squares   F-statistic:                     12.22
Date:                Wed, 18 Feb 2026   Prob (F-statistic):           5.76e-05
Time:                        22:52:01   Log-Likelihood:                 10.089
No. Observations:                  48   AIC:                            -14.18
Df Residuals:                      45   BIC:                            -8.564
Df Model:                           2                                         
Covariance Type:                  HC1                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                9.8950 

In [ ]:
#-------------------------------------
# Q4) Let's see all models side-by-side  
#-------------------------------------

from statsmodels.iolib.summary2 import summary_col

summary = summary_col(
    [ols, stage1, stage2],
    stars=True,
    model_names=["OLS", "First Stage", "Second Stage"],
    info_dict={"N":lambda x: f"{int(x.nobs)}"}
)

print(summary)



                        OLS     First Stage Second Stage
--------------------------------------------------------
Intercept            10.3420*** 4.1030***   9.8950***   
                     (0.9665)   (0.0884)    (1.0973)    
log_rprice           -1.4065***                         
                     (0.2609)                           
log_rincome          0.3439     0.1083***   0.2804      
                     (0.2604)   (0.0397)    (0.2708)    
tdiff                           0.0109***               
                                (0.0021)                
tax_cpi                         0.0094***               
                                (0.0009)                
predicted_log_rprice                        -1.2774***  
                                            (0.2857)    
R-squared            0.4327     0.9403      0.3368      
R-squared Adj.       0.4075     0.9363      0.3073      
N                    48         48          48          
Standard errors in parentheses

In [13]:
print(iv_model.summary)


                          IV-2SLS Estimation Summary                          
Dep. Variable:              log_packs   R-squared:                      0.4294
Estimator:                    IV-2SLS   Adj. R-squared:                 0.4041
No. Observations:                  48   F-statistic:                    34.506
Date:                Wed, Feb 18 2026   P-value (F-stat)                0.0000
Time:                        22:51:54   Distribution:                  chi2(2)
Cov. Estimator:                robust                                         
                                                                              
                              Parameter Estimates                              
             Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------
Intercept       9.8950     0.9288     10.654     0.0000      8.0746      11.715
log_rincome     0.2804     0.2458     1.1407    

In [16]:
#---------------------------------------------------------
# Example 2: The effect of going to college on wages.  
#---------------------------------------------------------

# Now can you replicate the above for: The effect of college on wages. 
# Get data from Cunningham (2020)
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
def read_data(file): 
    return pd.read_stata("https://github.com/scunning1975/mixtape/raw/master/" + file)

card = read_data("card.dta")

# Complete the questions from lecture slide. 
